In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.utils import to_categorical

なんか色々ライブラリを読み込んでる。

In [ ]:
# CSV読み込み
train_df = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')
test_df = pd.read_csv('/kaggle/input/digit-recognizer/test.csv')

# 画像とラベルを分ける
train_images = train_df.drop('label', axis=1).values / 255.0  # 0-1正規化
train_labels = train_df['label'].values
test_images = test_df.values / 255.0  # 0-1正規化

# CNN用に形を変更（28x28x1）
X_train_cnn = train_images.reshape(-1,28,28,1)
X_test_cnn = test_images.reshape(-1,28,28,1)

# ラベルをone-hotエンコーディング
y_train_cat = to_categorical(train_labels, 10)

In [ ]:
model = Sequential([
    Conv2D(32, kernel_size=(5,5), activation='relu', input_shape=(28,28,1)),
    MaxPooling2D(pool_size=(2,2)),
    Conv2D(64, kernel_size=(3,3), activation='relu'),
    MaxPooling2D(pool_size=(2,2)),
    Flatten(),
    Dense(256, activation='relu'),
    Dense(10, activation='softmax')  # 出力層：10クラス
])

In [ ]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
model.fit(X_train_cnn, y_train_cat, epochs=40, batch_size=128, validation_split=0.1)

In [ ]:
preds = model.predict(X_test_cnn)
preds_classes = np.argmax(preds, axis=1)  # 確率の最大値を整数ラベルに変換

In [ ]:
"""
# 4. 提出用DataFrame作成
submission = pd.DataFrame({
    "ImageId": range(1, len(y_pred)+1),
    "Label": y_pred
})

# 5. CSV出力
submission.to_csv("submission.csv", index=False)
"""

# ===== 提出ファイル作成 =====
submission = pd.DataFrame({
    "ImageId": np.arange(1, len(preds)+1),
    "Label": preds_classes
})
submission_path = "/kaggle/working/submission.csv"
submission.to_csv(submission_path, index=False)
print("Saved:", submission_path)
submission.head()